In [174]:
from pathlib import Path
import re
import datetime

# ===== Experiment setup =====
MODEL = "llama3.3-70b"
TEMPERATURE = 0
CONFIG_NAME = "C-Min"

scenario_name = "Flashlight_functional"

iteration_id = 0

def slugify(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return re.sub(r"_+", "_", s).strip("_")

# ===== Create log file =====
logs_dir = Path("logs")
logs_dir.mkdir(exist_ok=True)

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
txt_path = logs_dir / f"{slugify(scenario_name)}__{slugify(CONFIG_NAME)}__{stamp}.txt"
txt_path= logs_dir / f"flashlight_functional__c_min__20260305-121114.txt"
"""with txt_path.open("w", encoding="utf-8") as f:
    f.write("EXPERIMENT LOG\n")
    f.write("=" * 70 + "\n")
    f.write(f"SCENARIO: {scenario_name}\n")

    f.write(f"CONFIG: {CONFIG_NAME}\n")
    f.write(f"MODEL: {MODEL}\n")
    f.write(f"TEMPERATURE: {TEMPERATURE}\n")
    f.write(f"CREATED_AT: {stamp}\n")
    f.write("=" * 70 + "\n\n")"""

print("Log file created:", txt_path)


Log file created: logs/flashlight_functional__c_min__20260305-121114.txt


In [175]:
def dict_to_bullets(d, indent: int = 4) -> str:
    """
    Turn a nested dict (and lists) into indented bullet lines.

    Change vs previous version:
    - FIRST-LEVEL keys are NOT printed (e.g., 'arch', 'relation', 'cara' won't appear).
    - Content under each first-level key is printed directly.
    """
    pad = " " * indent
    lines = []

    if d is None or d == {}:
        return f"{pad}- (none)"

    def walk(node, level: int, show_key: bool = True):
        pad2 = " " * level

        if isinstance(node, dict):
            if not node:
                lines.append(f"{pad2}- (none)")
                return
            for k, v in node.items():
                # Skip printing the first-level key labels
                if show_key:
                    key = str(k) if k is not None and str(k).strip() else "(unnamed)"
                    lines.append(f"{pad2}- {key}:")
                    walk(v, level + 2, show_key=True)
                else:
                    # First level: don't print key, just descend into its value
                    walk(v, level, show_key=True)

        elif isinstance(node, (list, tuple, set)):
            node = list(node)
            if not node:
                lines.append(f"{pad2}- (none)")
            else:
                for item in node:
                    if isinstance(item, (dict, list, tuple, set)):
                        lines.append(f"{pad2}-")
                        walk(item, level + 2, show_key=True)
                    else:
                        lines.append(f"{pad2}- {item}")

        else:
            val = str(node).strip()
            lines.append(f"{pad2}- {val if val else '(empty)'}")

    # Start at root with show_key=False so first-level keys are hidden
    walk(d, indent, show_key=False)
    return "\n".join(lines)


In [198]:
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local-dummy")

def generate_completion(
    scenario_name: str,
    elements: dict,
    model: str,
    temperature: float = 0.0,
    n_additions: int = 3,
    dsl: str = "sysmlv2",
) -> tuple[str, str]:
    

    # Build partial model text from elements
    elems_block =  dict_to_bullets(elements, indent=4)
    
    partial_model = f"""Elements: \n {elems_block}"""

    prompt = f"""
                You are a modeling assistant. Complete a partial {dsl} for the scenario.

                Scenario: {scenario_name}
                Current partial model:
                {partial_model}

                Task:
                Propose up to {n_additions} additions to extend the model. An addition should mention the type of the suggested concept as used in the metamodel of {dsl}
                and then you can optionally say if it is a characteristic or a relation or a architectural element.

                Write as a bullet list. Be concise. inspire from what is already in the model and follow the elements types there. Try to follow how things are linked and named in partial model and find what is missing.
                Try to connect elements and types that are already in the model. suggest existing types like in example  """.strip()

    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )

    output_text = resp.choices[0].message.content
    return prompt, output_text






In [ ]:
iteration_id=43

In [229]:
iteration_id += 1
prompt, output_text = generate_completion(
    scenario_name=scenario_name,
    elements = {
        "arch": {"PartDefinition": {"Flashlight", "Battery", "Mount", "Accent", "Switch", "Led ", "BatteryHolder","Lens", "glow", "Reflector"},
                   "RequirementDefinition":{"Light", "Glow", "Torch", "Lamp", "Beacon"},
                   "ConstraintDefinition": {"LightOnlyWhenOn"},
                     "PortDefinition": {"PowerPort"}

        },
        "characteristics": {
           "partUsage":{"battery", "Reflector","accent", "Led","BatteryHolder", "switch", "Lens", "focus","Glow", "Mount"},
             "RequirementUsage":{"battery, Light", "Beam", "Beacon"},
             "portUsage":{"battery.pwr", "lens.out","sw.in", "sw.out","led.pwr"},
             "ConstraintUsage": {"LightOnlyWhenOn"}
        },
        "connections": {
              "connections: partUsage - partDefinition : Accent -isTypedBy- Accent",
              "connections: partUsage - partDefinition : Led -isTypedBy- Led",
              "connections: partUsage - partDefinition : Glow -isTypedBy- Glow",
              "connections: partUsage - partDefinition : BatteryHolder -isTypedBy- BatteryHolder" ,
              "connections: partUsage - partDefinition : Reflector -isTypedBy- Reflector",
              "connection: partUsage - partDefinition: Switch -isTypedBy- Switch",
                "connection: RequirementUsage - RequirementDefinition : Light -isTypedBy- Light",
                 "connection:partUsage - partDefinition: Mount -isTypedBy- Mount",
                 "connection:partUsage - partDefinition : Lens -isTypedBy- Lens",
                 "partUsage - partDefinition': 'focus -isTypedBy- Focus"




              
             }
}
,
    model=MODEL,
    temperature=TEMPERATURE,
    n_additions=3,
    dsl="sysmlv2"
)

print("===== PROMPT =====")
print(prompt)
print("\n===== MODEL OUTPUT =====")
print(output_text)


===== PROMPT =====
You are a modeling assistant. Complete a partial sysmlv2 for the scenario.

                Scenario: Flashlight_functional
                Current partial model:
                Elements: 
     - PartDefinition:
      - Battery
      - Accent
      - Reflector
      - Led 
      - Mount
      - glow
      - Lens
      - Switch
      - Flashlight
      - BatteryHolder
    - RequirementDefinition:
      - Lamp
      - Glow
      - Beacon
      - Light
      - Torch
    - ConstraintDefinition:
      - LightOnlyWhenOn
    - PortDefinition:
      - PowerPort
    - partUsage:
      - Reflector
      - focus
      - switch
      - Led
      - battery
      - Lens
      - accent
      - Glow
      - Mount
      - BatteryHolder
    - RequirementUsage:
      - Beacon
      - Beam
      - battery, Light
    - portUsage:
      - battery.pwr
      - sw.out
      - sw.in
      - led.pwr
      - lens.out
    - ConstraintUsage:
      - LightOnlyWhenOn
    - connections: partUsage -

In [230]:
# ===== Manually fill this dict (we copy/paste items from output) =====
classified = {
    "arch": [
       
    ],
    "connections": [

       
    ],
    "characteristics": [
        {"ConstraintDefinition: ThermalLimit     "}
    ],
}


with txt_path.open("a", encoding="utf-8") as f:
    f.write(f"\n\nITERATION {iteration_id}\n")
    f.write("-" * 80 + "\n")



    f.write("PROMPT\n")
    f.write("-" * 80 + "\n")
    f.write(prompt + "\n\n")

    f.write("MODEL OUTPUT\n")
    f.write("-" * 80 + "\n")
    f.write(output_text.strip() + "\n\n")

    f.write("MANUAL CLASSIFICATION\n")
    f.write("-" * 80 + "\n")
    f.write("ARCH:\n")
    for x in classified["arch"]:
        f.write(f"- {x}\n")
    f.write("\nCONNECTIONS:\n")
    for x in classified["connections"]:
        f.write(f"- {x}\n")
    f.write("\nCHARACTERISTICS:\n")
    for x in classified["characteristics"]:
        f.write(f"- {x}\n")

print(f"\nLogged iteration {iteration_id} to:", txt_path)
classified


Logged iteration 45 to: logs/flashlight_functional__c_min__20260305-121114.txt


{'arch': [],
 'connections': [],
 'characteristics': [{'ConstraintDefinition: ThermalLimit     '}]}

In [231]:
# ===== Manually fill accepted items for this iteration =====
accepted = {
    "arch": [
      
    ],
    "connections": [
    ],
    "characteristics": [
       {""}
    ],
}

# ===== Append accepted to the same log file =====
with txt_path.open("a", encoding="utf-8") as f:
    f.write("\n\nACCEPTED (ITERATION {})\n".format(iteration_id))
    f.write("-" * 80 + "\n")

    f.write("ARCH:\n")
    for x in accepted["arch"]:
        f.write(f"- {x}\n")

    f.write("\nCONNECTIONS:\n")
    for x in accepted["connections"]:
        f.write(f"- {x}\n")

    f.write("\nCHARACTERISTICS:\n")
    for x in accepted["characteristics"]:
        f.write(f"- {x}\n")

print(f"Accepted suggestions appended for iteration {iteration_id} to:", txt_path)
accepted


Accepted suggestions appended for iteration 45 to: logs/flashlight_functional__c_min__20260305-121114.txt


{'arch': [], 'connections': [], 'characteristics': [{''}]}

In [228]:

print("===== ADDED ELEMENTS MANUALLY =====")


added_elements ={
    "arch": [
          


    ],
    "connections": [

        {"portUsage": "led.pwr"}

    ],
    "characteristics": [

    ],
}
# ===== Append new added line  to the same log file =====
with txt_path.open("a", encoding="utf-8") as f:
    f.write("\n\nADDED ELEMENTS MANUALLY (ITERATION {})\n".format(iteration_id))
    f.write("-" * 80 + "\n")

    f.write("ARCH:\n")
    for x in added_elements["arch"]:
        f.write(f"- {x}\n")

    f.write("\nCONNECTIONS:\n")
    for x in added_elements["connections"]:
        f.write(f"- {x}\n")

    f.write("\nCHARACTERISTICS:\n")
    for x in added_elements["characteristics"]:
        f.write(f"- {x}\n")
   

===== ADDED ELEMENTS MANUALLY =====
